In [ ]:
import dataset as ds
from dataset import TensorToImg, ImgWrite, ImgRead, ImgToTensor
# dt = ds.Coco("/home/wanderer2414/coco2017/")
from torch.utils.data import DataLoader
from dataset2 import YOLODataset, collect_fn
import config
from torch import tensor, arange, float as tfloat, stack
import matplotlib.pyplot as plt
import matplotlib.patches as pat

In [ ]:
# import MyRCNN
# import torch
# dev = "cpu"
# model = MyRCNN.Model(device=torch.device(dev))
# model.model.load_state_dict(torch.load("bbx.pth", map_location=dev))
def count_parameters(model):
        return sum(p.numel() for p in model.parameters())

# print(count_parameters(model.model.color))

In [ ]:
import MyRCNN
import torch
device = config.DEVICE
model = MyRCNN.Model(bbx_epoches=4, cls_epoches=4)
model = model.to(device)
model.train()

[00:22:39] Epoch 0/1; Loss 10.480897903442383 ███████                                                566      /     4138

In [ ]:
# model.Evaluate()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as pat
from torch import sigmoid, softmax, zeros, bool as tbool
from dataset import TensorToImg, ImgToTensor

model.eval()
for i, (tens,label) in enumerate(model.loader):
    if (i==0): continue
    data = tens.to(device)
    boundary, score, bbx = model.model(data)
    color = model.model.color(data)
    prep = model.model.color.prepare(data)
    print(prep.shape)
    # x =score
    # x = color[:, 1:2,: ,:]
    # x = x.repeat(1, 3, 1, 1)
    # x = x-x.min()
    # x = x/x.max()
    x = tens
    x = TensorToImg(x.detach().cpu())
    plt.imshow(x)
    N = prep.shape[1]
    fig, axes = plt.subplots(N//4, 4, figsize=(12, 20))
    cls_range = arange(20).view(20, 1, 1).expand(20, N, 1)
    for i, ax in enumerate(axes.flat):
        x = prep[:, i:i+1, :, :]
        # print(x.min(), x.max())
        x = x.repeat(1, 3, 1, 1)
        x = x-x.min()
        # x = x/x.max()
        cls, x1, y1, x2, y2, sc = bbx[i].detach().cpu().numpy()
        print((x2 -x1), (y2-y1))
        rect = pat.Rectangle((x1, y1), x2-x1, y2-y1, facecolor='none', edgecolor='red')
        ax.add_patch(rect)
        ax.text((x1+x2)/2, (y1+y2)/2, f"{config.PASCAL_CLASSES[cls.astype(int)]}:{0}%", fontsize=12, color='red')
        ax.imshow(TensorToImg(x.detach().cpu()))
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    break